# Module 2: Data Modeling and Performance with Python

Run this notebook on the provided workshop VM after completing Module 1. It uses `AzureCliCredential`, creates deterministic lab data, compares embedded and polymorphic models, and measures query plans before and after ESR indexes.

> This notebook replaces the shell-based Module 2 exercises. Run the cells in order. It resets only the `customers`, `orders`, and `demo` collections in the `docdbworkshop` database.


In [ ]:
import os
from datetime import datetime, timedelta, timezone

from azure.identity import AzureCliCredential
from pymongo import MongoClient
from pymongo.auth_oidc import OIDCCallback, OIDCCallbackContext, OIDCCallbackResult

DOCUMENTDB_SCOPE = "https://ossrdbms-aad.database.windows.net/.default"

class AzureIdentityTokenCallback(OIDCCallback):
    def __init__(self, credential):
        self.credential = credential

    def fetch(self, context: OIDCCallbackContext) -> OIDCCallbackResult:
        del context
        token = self.credential.get_token(DOCUMENTDB_SCOPE)
        return OIDCCallbackResult(access_token=token.token)

cluster_name = os.environ["DOCUMENTDB_CLUSTER_NAME"]
credential = AzureCliCredential()
client = MongoClient(
    f"mongodb+srv://{cluster_name}.global.mongocluster.cosmos.azure.com/",
    tls=True,
    retryWrites=False,
    authMechanism="MONGODB-OIDC",
    authMechanismProperties={"OIDC_CALLBACK": AzureIdentityTokenCallback(credential)},
)
db = client["docdbworkshop"]
customers = db["customers"]
orders = db["orders"]
demo = db["demo"]
print(db.command({"ping": 1}))


## Step 1: Load deterministic data

The generated values are identical across Python, C#, and Node.js. This makes explain-plan comparisons repeatable. Expected counts are 10 customers and 10,000 orders.


In [ ]:
statuses = ["shipped", "pending", "delivered", "cancelled", "processing"]
skus = ["SKU-A1", "SKU-B2", "SKU-C3", "SKU-D4", "SKU-E5", "SKU-F6", "SKU-G7", "SKU-H8"]
start = datetime(2023, 1, 1, tzinfo=timezone.utc)

customer_documents = [
    {
        "_id": f"C-{1001 + index}",
        "name": f"Workshop Customer {index + 1}",
        "tier": ["gold", "silver", "bronze"][index % 3],
        "region": ["eastus", "westus", "central"][index % 3],
    }
    for index in range(10)
]
order_documents = [
    {
        "_id": f"O-{index + 1:05d}",
        "customerId": f"C-{1001 + index % 10}",
        "status": statuses[index * 7 % len(statuses)],
        "total": ((index * 7919) % 99900 + 100) / 100,
        "createdAt": start + timedelta(days=index * 37 % 730),
        "items": [{"sku": skus[index * 3 % len(skus)], "qty": index % 5 + 1}],
    }
    for index in range(10_000)
]

customers.delete_many({})
orders.delete_many({})
demo.delete_many({})
customers.insert_many(customer_documents)
for offset in range(0, len(order_documents), 1000):
    orders.insert_many(order_documents[offset:offset + 1000])

{"customers": customers.count_documents({}), "orders": orders.count_documents({})}


## Step 2: Compare embedded and polymorphic models

The embedded order is one document and supports a single-read aggregate view. The polymorphic model stores one header plus separate line-item documents, which avoids an unbounded items array but requires a multi-document query.


In [ ]:
demo.insert_many([
    {
        "_id": "DEMO-001",
        "type": "embedded_order",
        "customerId": "C-1001",
        "total": 79.90,
        "items": [{"sku": "BOOK-101", "qty": 1}, {"sku": "BOOK-202", "qty": 1}],
    },
    {"_id": "DEMO-002", "orderId": "DEMO-002", "type": "order", "customerId": "C-1001", "total": 79.90},
    {"_id": "DEMO-002-1", "orderId": "DEMO-002", "type": "line_item", "sku": "BOOK-101", "qty": 1},
    {"_id": "DEMO-002-2", "orderId": "DEMO-002", "type": "line_item", "sku": "BOOK-202", "qty": 1},
])

{
    "embeddedDocuments": demo.count_documents({"_id": "DEMO-001"}),
    "polymorphicDocuments": demo.count_documents({"orderId": "DEMO-002"}),
    "embeddedOrder": demo.find_one({"_id": "DEMO-001"}),
    "polymorphicOrder": list(demo.find({"orderId": "DEMO-002"}, {"_id": 1, "type": 1, "sku": 1})),
}


In [ ]:
status_counts = list(orders.aggregate([
    {"$group": {"_id": "$status", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
]))
status_counts


## Step 3: Compare baseline and indexed plans

For each query, compare `totalDocsExamined`, `totalKeysExamined`, and execution time. The first index follows equality, sort/range ordering for the finance query. The second demonstrates a separate sort field followed by the remaining range field.


In [ ]:
def explain_find(filter_document, sort_document):
    return db.command({
        "explain": {"find": "orders", "filter": filter_document, "sort": sort_document},
        "verbosity": "executionStats",
    })


def summarize_explain(label, explain):
    stats = explain.get("executionStats", explain)
    return {
        "plan": label,
        "stage": explain.get("queryPlanner", {}).get("winningPlan", {}).get("stage", "see full plan"),
        "nReturned": stats.get("nReturned"),
        "totalDocsExamined": stats.get("totalDocsExamined"),
        "totalKeysExamined": stats.get("totalKeysExamined"),
        "executionTimeMillis": stats.get("executionTimeMillis"),
    }

orders.drop_indexes()
finance_filter = {"status": "delivered", "total": {"$gte": 500}}
finance_baseline = explain_find(finance_filter, {"total": -1})
orders.create_index([("status", 1), ("total", -1)], name="status_1_total_-1")
finance_indexed = explain_find(finance_filter, {"total": -1})
[
    summarize_explain("finance baseline", finance_baseline),
    summarize_explain("finance ESR index", finance_indexed),
]


In [ ]:
orders.drop_indexes()
customer_filter = {
    "customerId": "C-1006",
    "status": "shipped",
    "total": {"$gte": 100},
}
customer_baseline = explain_find(customer_filter, {"createdAt": -1})
orders.create_index(
    [("customerId", 1), ("status", 1), ("createdAt", -1), ("total", 1)],
    name="customer_status_date_total",
)
customer_indexed = explain_find(customer_filter, {"createdAt": -1})
[
    summarize_explain("customer baseline", customer_baseline),
    summarize_explain("customer ESR index", customer_indexed),
]


## Step 4: Run the leaderboard aggregation

The index can reduce documents entering the grouping stage, but it cannot eliminate the final sort because `totalRevenue` is computed by `$group` and does not exist in stored documents.


In [ ]:
leaderboard = list(orders.aggregate([
    {"$match": {"status": "shipped", "createdAt": {"$gte": datetime(2024, 1, 1, tzinfo=timezone.utc)}}},
    {"$group": {"_id": "$customerId", "totalRevenue": {"$sum": "$total"}, "orderCount": {"$sum": 1}}},
    {"$sort": {"totalRevenue": -1}},
    {"$limit": 5},
]))
leaderboard


## Lab success check

* [ ] The dataset contains 10 customers and 10,000 orders.
* [ ] You can explain why the embedded model uses one document and the polymorphic model uses three.
* [ ] You compared `totalDocsExamined` before and after both ESR indexes.
* [ ] You can explain why an index cannot remove the leaderboard sort on computed `totalRevenue`.
* [ ] The leaderboard returns up to five customers.
